In [1]:
try:
    spark.stop()
    print("✅ Session existante arrêtée")
except NameError:
    print("Aucune session existante trouvée")

✅ Session existante arrêtée


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bronze-Ingestion") \
    .config("spark.sql.catalog.demo.type", "rest") \
    .config("spark.sql.catalog.demo.uri", "http://rest:8181") \
    .config("spark.sql.catalog.demo.warehouse", "s3://lakehouse/") \
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.demo.s3.path-style-access", "true") \
    .config("spark.sql.catalog.demo.s3.access-key-id", "adminn") \
    .config("spark.sql.catalog.demo.s3.secret-access-key", "password") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "adminn") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS bronze")
print("✅ Session Spark prête, namespace bronze créé")

✅ Session Spark prête, namespace bronze créé


In [3]:
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name

# Lecture depuis le volume local monté (au lieu de s3a://)
df_json = spark.read.option("multiLine", "true").json("/home/iceberg/data/raw/inspection-app/")

df_bronze_missions = df_json \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_missions.printSchema()
df_bronze_missions.show(truncate=False)

df_bronze_missions.writeTo("demo.bronze.inspection_missions").createOrReplace()

print("✅ Table bronze.inspection_missions créée")

In [ ]:
from pyspark.sql.functions import current_timestamp, input_file_name

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv("s3a://landing-zone/sensors/")

df_bronze_sensors = df_csv \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_sensors.printSchema()
df_bronze_sensors.show()

df_bronze_sensors.writeTo("demo.bronze.sensor_readings").createOrReplace()

print("✅ Table bronze.sensor_readings créée")

In [ ]:
df_geojson_raw = spark.read.option("multiLine", "true").json("/home/iceberg/data/raw/hbim/")

df_bronze_hbim = df_geojson_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

df_bronze_hbim.printSchema()
df_bronze_hbim.show(truncate=False)

df_bronze_hbim.writeTo("demo.bronze.hbim_zones").createOrReplace()

print("✅ Table bronze.hbim_zones créée")

In [ ]:
spark.sql("SHOW TABLES IN demo.bronze").show()

In [ ]:
from pyspark.sql.functions import explode, col, to_date, trim, initcap, upper, when

df_bronze_missions = spark.table("demo.bronze.inspection_missions")

df_silver_anomalies = df_bronze_missions \
    .select(
        # Extraire les champs de la mission
        col("mission_id"),
        col("inspection_date"),
        col("inspector_id"),
        col("zone_id"),
        # Déplier la liste des anomalies -> une ligne par anomalie
        explode(col("anomalies")).alias("anomaly")
    ) \
    .select(
        col("mission_id"),
        # Convertir inspection_date en format de date standard
        to_date(col("inspection_date"), "yyyy-MM-dd").alias("inspection_date"),
        col("inspector_id"),
        col("zone_id"),
        col("anomaly.anomaly_id").alias("anomaly_id"),
        col("anomaly.element_id").alias("element_id"),
        # Normaliser le type d'anomalie (espaces retirés, casse cohérente)
        initcap(trim(col("anomaly.type"))).alias("anomaly_type"),
        # Normaliser la criticité (majuscules, cohérent pour les filtres/graphiques)
        upper(trim(col("anomaly.criticality"))).alias("criticality")
    ) \
    .filter(
        # Vérifier les identifiants : aucun ne doit être vide/null
        col("mission_id").isNotNull() &
        col("zone_id").isNotNull() &
        col("anomaly_id").isNotNull() &
        col("element_id").isNotNull()
    )

df_silver_anomalies.show(truncate=False)

df_silver_anomalies.writeTo("demo.silver.inspection_anomalies").createOrReplace()

print("✅ silver.inspection_anomalies —", df_silver_anomalies.count(), "anomalies valides")

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    to_timestamp, to_date as to_date_col, col, when, lit,
    last, first, percentile_approx, count as spark_count
)

# --- 1. Lecture Bronze ---
df_bronze_sensors = spark.table("demo.bronze.sensor_readings")

# --- 2. Typage du timestamp + colonne date ---
df = df_bronze_sensors \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("reading_date", to_date_col(col("timestamp")))

# --- 3. Suppression des doublons exacts (même capteur, même timestamp) ---
df = df.dropDuplicates(["sensor_id", "timestamp"])

# --- 4. Bornes physiques plausibles ---
TEMP_MIN, TEMP_MAX = -20.0, 60.0
HUMIDITY_MIN, HUMIDITY_MAX = 0, 100

# --- 5. Marquer les valeurs aberrantes comme manquantes (null) ---
df = df.withColumn(
    "temperature",
    when((col("temperature") < TEMP_MIN) | (col("temperature") > TEMP_MAX), None)
    .otherwise(col("temperature"))
).withColumn(
    "humidity",
    when((col("humidity") < HUMIDITY_MIN) | (col("humidity") > HUMIDITY_MAX), None)
    .otherwise(col("humidity"))
)

# --- 6. Supprimer les lignes où les identifiants essentiels manquent (non récupérables) ---
df = df.filter(
    col("sensor_id").isNotNull() &
    col("zone_id").isNotNull() &
    col("timestamp").isNotNull()
)

# --- 7. Supprimer les lignes où TOUTES les mesures sont manquantes en même temps (ligne inexploitable) ---
df = df.filter(~(col("temperature").isNull() & col("humidity").isNull()))

# --- 8. Interpolation temporelle par zone : combler avec la valeur voisine (avant ou après) ---
window_zone = Window.partitionBy("zone_id").orderBy("timestamp")

df = df.withColumn(
    "temperature_filled",
    when(col("temperature").isNotNull(), col("temperature"))
    .otherwise(last("temperature", ignorenulls=True).over(window_zone.rowsBetween(Window.unboundedPreceding, 0)))
).withColumn(
    "humidity_filled",
    when(col("humidity").isNotNull(), col("humidity"))
    .otherwise(last("humidity", ignorenulls=True).over(window_zone.rowsBetween(Window.unboundedPreceding, 0)))
)

# --- 9. Pour les valeurs encore manquantes après interpolation (ex: début de série), utiliser la médiane de la zone ---
df_median = df.groupBy("zone_id").agg(
    percentile_approx("temperature", 0.5).alias("temp_median"),
    percentile_approx("humidity", 0.5).alias("humidity_median")
)

df = df.join(df_median, on="zone_id", how="left")

df = df.withColumn(
    "temperature_filled",
    when(col("temperature_filled").isNotNull(), col("temperature_filled"))
    .otherwise(col("temp_median"))
).withColumn(
    "humidity_filled",
    when(col("humidity_filled").isNotNull(), col("humidity_filled"))
    .otherwise(col("humidity_median"))
)

# --- 10. Colonne de traçabilité : quel type de correction a été appliqué ---
df = df.withColumn(
    "correction_type",
    when(col("temperature").isNotNull() & col("humidity").isNotNull(), lit("aucune"))
    .when(col("temperature_filled") == col("temp_median"), lit("median"))
    .otherwise(lit("interpolation"))
)

# --- 11. Nettoyage final : garder les colonnes utiles, renommer proprement ---
df_silver_sensors = df.select(
    "sensor_id",
    "zone_id",
    "timestamp",
    "reading_date",
    col("temperature_filled").alias("temperature"),
    col("humidity_filled").alias("humidity"),
    "correction_type",
    "ingestion_timestamp",
    "source_file"
)

df_silver_sensors.show(20, truncate=False)

df_silver_sensors.writeTo("demo.silver.sensor_readings_clean").createOrReplace()

nb_corrections = df_silver_sensors.filter(col("correction_type") != "aucune").count()
total = df_silver_sensors.count()
print(f"✅ silver.sensor_readings_clean — {total} lignes, dont {nb_corrections} corrigées (médiane/interpolation)")

In [ ]:
from pyspark.sql.functions import avg, min as spark_min, max as spark_max, round as spark_round

df_sensor_zone_daily = spark.table("demo.silver.sensor_readings_clean") \
    .groupBy("zone_id", "reading_date") \
    .agg(
        spark_round(avg("temperature"), 1).alias("temp_avg"),
        spark_round(spark_min("temperature"), 1).alias("temp_min"),
        spark_round(spark_max("temperature"), 1).alias("temp_max"),
        spark_round(avg("humidity"), 1).alias("humidity_avg"),
        spark_round(spark_min("humidity"), 1).alias("humidity_min"),
        spark_round(spark_max("humidity"), 1).alias("humidity_max")
    ) \
    .orderBy("zone_id", "reading_date")

df_sensor_zone_daily.show()

df_sensor_zone_daily.writeTo("demo.silver.sensor_zone_daily_stats").createOrReplace()

print("✅ silver.sensor_zone_daily_stats créée — indicateurs par zone/jour")

In [ ]:
from pyspark.sql.functions import size, expr

df_bronze_hbim = spark.table("demo.bronze.hbim_zones")

df_silver_hbim = df_bronze_hbim \
    .select(explode(col("features")).alias("feature")) \
    .select(
        # Harmoniser zone_id (espaces retirés, majuscules)
        upper(trim(col("feature.properties.zone_id"))).alias("zone_id"),
        col("feature.properties.zone_name").alias("zone_name"),
        col("feature.properties.level").alias("level"),
        col("feature.properties.material").alias("material"),
        col("feature.geometry.type").alias("geometry_type"),
        col("feature.geometry.coordinates").alias("coordinates")
    ) \
    .withColumn(
        # Valider la géométrie : doit être un Polygone avec au moins 4 points
        "geometry_valid",
        (col("geometry_type") == "Polygon") &
        (size(col("coordinates")[0]) >= 4)
    ) \
    .withColumn(
        # Vérifier le système de coordonnées : longitude/latitude dans les bornes valides (WGS84)
        "coordinates_valid",
        expr("""
            forall(coordinates[0], point ->
                point[0] >= -180 AND point[0] <= 180 AND
                point[1] >= -90 AND point[1] <= 90
            )
        """)
    ) \
    .filter(col("zone_id").isNotNull())

df_silver_hbim.show(truncate=False)

df_silver_hbim.writeTo("demo.silver.hbim_zones_clean").createOrReplace()

invalid = df_silver_hbim.filter(~col("geometry_valid") | ~col("coordinates_valid")).count()
print(f"✅ silver.hbim_zones_clean créée — {invalid} zone(s) avec géométrie/coordonnées invalides (à corriger si > 0)")

In [ ]:
spark.sql("SHOW TABLES IN demo.silver").show()

In [ ]:
df_anomalies = spark.table("demo.silver.inspection_anomalies")
df_hbim = spark.table("demo.silver.hbim_zones_clean")

df_gold_anomalies_enriched = df_anomalies.join(
    df_hbim.select("zone_id", "zone_name", "level", "material"),
    on="zone_id",
    how="left"
)

df_gold_anomalies_enriched.show(truncate=False)

df_gold_anomalies_enriched.writeTo("demo.gold.anomalies_enriched").createOrReplace()

print("✅ gold.anomalies_enriched — anomalies liées à leur zone géographique")

In [ ]:
from pyspark.sql.functions import count, countDistinct, lit

df_kpi = spark.sql("""
    SELECT
        COUNT(DISTINCT mission_id)   AS total_missions,
        COUNT(*)                     AS total_anomalies,
        SUM(CASE WHEN criticality IN ('ÉLEVÉE', 'CRITIQUE') THEN 1 ELSE 0 END) AS critical_anomalies,
        COUNT(DISTINCT zone_id)      AS zones_inspected
    FROM demo.silver.inspection_anomalies
""")

df_kpi.show()

df_kpi.writeTo("demo.gold.kpi_summary").createOrReplace()

print("✅ gold.kpi_summary créée")

In [ ]:
df_gold_by_type = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("anomaly_type") \
    .agg(count("*").alias("nb_anomalies")) \
    .orderBy(col("nb_anomalies").desc())

df_gold_by_type.show()
df_gold_by_type.writeTo("demo.gold.anomalies_by_type").createOrReplace()

df_gold_by_criticality = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("criticality") \
    .agg(count("*").alias("nb_anomalies")) \
    .orderBy(col("nb_anomalies").desc())

df_gold_by_criticality.show()
df_gold_by_criticality.writeTo("demo.gold.anomalies_by_criticality").createOrReplace()

print("✅ gold.anomalies_by_type et gold.anomalies_by_criticality créées")

In [ ]:
# Nombre d'anomalies par zone
df_anomalies_by_zone = spark.table("demo.silver.inspection_anomalies") \
    .groupBy("zone_id") \
    .agg(
        count("*").alias("nb_anomalies"),
        spark_round(
            avg(when(col("criticality").isin("ÉLEVÉE", "CRITIQUE"), 1).otherwise(0)) * 100, 1
        ).alias("pct_anomalies_critiques")
    )

# Moyenne température/humidité par zone (toutes périodes confondues)
df_env_by_zone = spark.table("demo.silver.sensor_zone_daily_stats") \
    .groupBy("zone_id") \
    .agg(
        spark_round(avg("temp_avg"), 1).alias("temp_moyenne"),
        spark_round(avg("humidity_avg"), 1).alias("humidite_moyenne")
    )

# Croisement final : zone géographique + anomalies + environnement
df_gold_zone_summary = df_hbim.select("zone_id", "zone_name", "level", "material") \
    .join(df_anomalies_by_zone, on="zone_id", how="left") \
    .join(df_env_by_zone, on="zone_id", how="left") \
    .fillna(0, subset=["nb_anomalies", "pct_anomalies_critiques"])

df_gold_zone_summary.show(truncate=False)

df_gold_zone_summary.writeTo("demo.gold.zone_summary").createOrReplace()

print("✅ gold.zone_summary — indicateur combiné complet par zone")

In [ ]:
spark.sql("SHOW TABLES IN demo.gold").show()

print("\n--- KPI globaux ---")
spark.table("demo.gold.kpi_summary").show()

print("\n--- Synthèse par zone (table clé du dashboard) ---")
spark.table("demo.gold.zone_summary").show(truncate=False)

In [ ]:
!pip install "opencv-python-headless==4.10.0.84" "numpy<2" pillow imagehash boto3 imgaug
print("✅ Dépendances installées")

In [ ]:
import boto3
import os
import hashlib
from datetime import datetime
from PIL import Image
from pyspark.sql import Row

# Client S3 pour transférer les fichiers binaires (Spark ne gère pas bien le binaire, boto3 est plus adapté)
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="adminn",
    aws_secret_access_key="password"
)

SOURCE_DIR = "/home/iceberg/data/raw/images"

# 1) Upload des fichiers bruts vers MinIO (zone Bronze)
for filename in os.listdir(SOURCE_DIR):
    filepath = os.path.join(SOURCE_DIR, filename)
    s3_client.upload_file(filepath, "lakehouse", f"bronze/images_raw/{filename}")

print("✅ Images brutes uploadées dans lakehouse/bronze/images_raw/")

# 2) Créer le registre de métadonnées (table Iceberg Bronze)
bronze_metadata = []

for filename in os.listdir(SOURCE_DIR):
    filepath = os.path.join(SOURCE_DIR, filename)
    try:
        with Image.open(filepath) as img:
            img.verify()
        with Image.open(filepath) as img:  # réouverture nécessaire après verify()
            width, height = img.size
            format_ = img.format

        with open(filepath, "rb") as f:
            content_hash = hashlib.sha256(f.read()).hexdigest()

        bronze_metadata.append(Row(
            filename=filename,
            s3_path=f"s3://lakehouse/bronze/images_raw/{filename}",
            width=width, height=height, format=format_,
            content_hash=content_hash, is_valid=True,
            ingestion_timestamp=datetime.now()
        ))
    except Exception:
        bronze_metadata.append(Row(
            filename=filename,
            s3_path=f"s3://lakehouse/bronze/images_raw/{filename}",
            width=None, height=None, format=None,
            content_hash=None, is_valid=False,
            ingestion_timestamp=datetime.now()
        ))

df_bronze_images = spark.createDataFrame(bronze_metadata)
df_bronze_images.show(truncate=False)

spark.sql("CREATE NAMESPACE IF NOT EXISTS bronze")
df_bronze_images.writeTo("demo.bronze.images_metadata").createOrReplace()

print("✅ Table bronze.images_metadata créée")

In [ ]:
from pyspark.sql.functions import udf, col, lit
from pyspark.sql.types import StringType

# Fonction de traitement : tous les imports sont à l'intérieur
def process_image(s3_path):
    try:
        import boto3
        import cv2
        import numpy as np
        import io
        import uuid
        from botocore.exceptions import ClientError

        # Créer le client S3 à chaque appel (nécessaire pour la sérialisation)
        s3_client = boto3.client(
            "s3",
            endpoint_url="http://minio:9000",
            aws_access_key_id="adminn",
            aws_secret_access_key="password"
        )

        # Extraire bucket et clé depuis s3_path
        parts = s3_path.replace("s3://", "").split("/", 1)
        if len(parts) != 2:
            return None
        bucket_name = parts[0]
        key = parts[1]

        # Télécharger l'image
        response = s3_client.get_object(Bucket=bucket_name, Key=key)
        img_bytes = response['Body'].read()
        np_arr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
        if img is None:
            return None

        # 1. Niveaux de gris
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 2. CLAHE (contraste local)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)

        # 3. Filtre bilatéral (réduction bruit)
        denoised = cv2.bilateralFilter(enhanced, 9, 75, 75)

        # 4. Redimensionnement 256x256
        resized = cv2.resize(denoised, (256, 256))

        # 5. Normalisation et conversion en uint8 pour JPEG
        normalized = (resized / 255.0 * 255).astype(np.uint8)

        # 6. Encodage JPEG et upload
        _, buffer = cv2.imencode('.jpg', normalized)
        new_key = f"silver/images_standardized_v2/{uuid.uuid4().hex}.jpg"
        s3_client.put_object(
            Bucket=bucket_name,
            Key=new_key,
            Body=io.BytesIO(buffer.tobytes()),
            ContentType='image/jpeg'
        )
        new_s3_path = f"s3://{bucket_name}/{new_key}"
        return new_s3_path

    except Exception as e:
        # Loguer l'erreur (vous pouvez aussi la collecter)
        print(f"Erreur sur {s3_path}: {e}")
        return None

# Déclarer l'UDF Spark
process_udf = udf(process_image, StringType())

# Lire la table Bronze
df_bronze = spark.table("demo.bronze.images_metadata")

# Appliquer le traitement
df_silver_v2 = (
    df_bronze
    .filter(col("is_valid") == True)
    .withColumn("processed_s3_path", process_udf(col("s3_path")))
    .filter(col("processed_s3_path").isNotNull())
)

# Ajouter des métadonnées
df_silver_v2 = (
    df_silver_v2
    .withColumn("width", lit(256))
    .withColumn("height", lit(256))
    .withColumn("format", lit("JPEG"))
    .withColumn("preprocessing", lit("gray+clahe+bilateral+resize256"))
)

# Écrire la table Silver V2
df_silver_v2.writeTo("demo.silver.images_standardized_v2").createOrReplace()

print("✅ Table silver.images_standardized_v2 créée avec images prétraitées.")

In [ ]:
from pyspark.sql.functions import lit

# Sélectionner les colonnes nécessaires et ajouter une colonne label vide
df_to_label = spark.table("demo.silver.images_standardized_v2") \
    .select("filename", "processed_s3_path") \
    .withColumn("label", lit(""))

# Écrire en CSV (un seul fichier pour faciliter l'édition)
df_to_label.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("s3a://lakehouse/temp/images_to_label/")

print("✅ Fichier CSV créé dans s3a://lakehouse/temp/images_to_label/")
print("   Téléchargez-le depuis MinIO, remplissez la colonne 'label' (crack/no_crack), puis re-téléversez-le.")

In [ ]:
# Lire le CSV labellisé depuis MinIO
df_labels = spark.read.option("header", "true") \
    .csv("s3a://lakehouse/temp/images_to_label/")

# Nettoyer les noms de colonnes (enlever les espaces)
df_labels = df_labels.toDF(*[c.strip() for c in df_labels.columns])

# Joindre avec la table Silver V2 en qualifiant les colonnes
df_silver = spark.table("demo.silver.images_standardized_v2")
df_labeled = df_silver.join(df_labels, on="filename", how="inner") \
    .select(df_silver["filename"], df_silver["processed_s3_path"], df_labels["label"])

# Filtrer les lignes où le label n'est pas vide
from pyspark.sql.functions import col
df_labeled = df_labeled.filter(col("label") != "")

# Sauvegarder la table labellisée
df_labeled.writeTo("demo.silver.images_labeled").createOrReplace()
print("✅ Table silver.images_labeled créée avec les labels.")
print(f"Nombre d'images labellisées : {df_labeled.count()}")

In [ ]:
from pyspark.sql.functions import rand, when, lit

# Générer des labels aléatoires pour tester
df_test_labels = spark.table("demo.silver.images_standardized_v2") \
    .select("filename", "processed_s3_path") \
    .withColumn("label", when(rand() > 0.5, lit("crack")).otherwise(lit("no_crack")))

df_test_labels.writeTo("demo.silver.images_labeled").createOrReplace()
print("✅ Labels aléatoires générés pour test.")

In [ ]:
import cv2
import numpy as np

print("cv2 :", cv2.__version__)
print("numpy :", np.__version__)

test_img = np.zeros((100, 100), dtype=np.uint8)
M = cv2.getRotationMatrix2D((50, 50), 10, 1)
out = cv2.warpAffine(test_img, M, (100, 100))
print("✅ OpenCV fonctionne correctement avec numpy.")

In [ ]:
import random
import uuid
import io
import boto3
import cv2
import numpy as np
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp

# Client S3
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="adminn",
    aws_secret_access_key="password"
)

# Lire la table labellisée
df_labeled = spark.table("demo.silver.images_labeled")
print("Schéma de images_labeled :")
df_labeled.printSchema()

rows = df_labeled.collect()
if len(rows) == 0:
    print("❌ Aucune image labellisée. Veuillez d'abord importer les labels.")
else:
    random.seed(42)
    random.shuffle(rows)
    n = len(rows)
    train_rows = rows[:int(n*0.7)]
    val_rows = rows[int(n*0.7):int(n*0.85)]
    test_rows = rows[int(n*0.85):]

    def augment_image(img):
        """Applique des transformations aléatoires à une image en niveaux de gris (numpy array uint8)."""
        # Copie explicite, contiguë et modifiable (obligatoire pour OpenCV 5.x)
        img = np.array(img, dtype=np.uint8, copy=True, order='C')

        if random.random() > 0.5:
            img = np.array(cv2.flip(img, 1), dtype=np.uint8, copy=True, order='C')

        angle = random.uniform(-10, 10)
        h, w = int(img.shape[0]), int(img.shape[1])
        M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        img = np.array(img, dtype=np.uint8, copy=True, order='C')

        factor = random.uniform(0.8, 1.2)
        img = np.clip(img * factor, 0, 255).astype(np.uint8)

        if random.random() > 0.3:
            noise = np.random.normal(0, 5, img.shape).astype(np.uint8)
            img = np.clip(img.astype(np.int16) + noise.astype(np.int16), 0, 255).astype(np.uint8)

        return np.array(img, dtype=np.uint8, copy=True, order='C')

    def upload_augmented(row, split, aug_count=3):
        s3_path = row["processed_s3_path"]
        parts = s3_path.replace("s3://", "").split("/", 1)
        if len(parts) != 2:
            return []
        bucket = parts[0]
        key = parts[1]
        try:
            response = s3_client.get_object(Bucket=bucket, Key=key)
            img_bytes = response['Body'].read()
            if not img_bytes:
                return []
            # Décodage direct avec OpenCV (gère nativement le JPG)
            # -> retourne toujours un tableau numpy contigu et modifiable
            nparr = np.frombuffer(img_bytes, dtype=np.uint8).copy()
            img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)
            if img is None or img.size == 0:
                print(f"⚠️ Image illisible : {s3_path}")
                return []
        except Exception as e:
            print(f"Erreur chargement {s3_path}: {e}")
            return []

        metadata_list = []
        for i in range(aug_count):
            aug_img = augment_image(img)
            ok, buffer = cv2.imencode('.jpg', aug_img)
            if not ok:
                print(f"⚠️ Échec d'encodage pour {row['filename']} (aug {i})")
                continue
            new_key = f"gold/dataset/{split}/{uuid.uuid4().hex}.jpg"
            try:
                s3_client.put_object(
                    Bucket=bucket,
                    Key=new_key,
                    Body=io.BytesIO(buffer.tobytes()),
                    ContentType='image/jpeg'
                )
            except Exception as e:
                print(f"Erreur upload {new_key}: {e}")
                continue
            new_s3_path = f"s3://{bucket}/{new_key}"
            meta = {
                "filename": f"{row['filename']}_aug{i}",
                "s3_path": new_s3_path,
                "label": row["label"],
                "split": split,
                "augmentation_id": i,
                "source_filename": row["filename"]
            }
            metadata_list.append(meta)
        return metadata_list

    # Générer les métadonnées Gold
    gold_metadata = []
    for split, rows_split in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
        for row in rows_split:
            aug_count = 5 if split == "train" else 1
            gold_metadata.extend(upload_augmented(row, split, aug_count))

    if gold_metadata:
        gold_rows = [Row(**meta) for meta in gold_metadata]
        df_gold = spark.createDataFrame(gold_rows)
        df_gold = df_gold.withColumn("dataset_version", lit("v1_crack_detection")) \
            .withColumn("created_at", current_timestamp())
        df_gold.writeTo("demo.gold.images_dataset_manifest_v2").createOrReplace()
        print("✅ Table gold.images_dataset_manifest_v2 créée.")
        print(f"Nombre total d'images : {df_gold.count()}")
        df_gold.groupBy("split").count().show()
    else:
        print("❌ Aucune métadonnée générée. Vérifiez les labels et les chemins.")